Здесь я экспериментирую с feature extraction. Я и тренирую здесь простые модели, но только для оценки полезности признаков, разработкой собственно модели можно будет заняться только позднее.

In [1]:
import polars as pl
dataset = pl.read_csv(
    '../data/processed/dataset_clean_onehot_label.csv'
)


In [2]:
SEED = 42


In [3]:
dataset.columns


['StimSite',
 'Stimulus',
 'RT_start',
 'Response_annot',
 'Response_transcription_annot',
 'speech arrest',
 'аномия',
 'дизартрия',
 'задержка',
 'нет',
 'поиск слова',
 'семантическая парафазия',
 'фонетическая парафазия',
 'Error_type_annot',
 'error_type_label']

In [4]:
dataset_0 = dataset[[
    'Stimulus',
    'RT_start',
    'Response_annot',
    'error_type_label'
    ]]\

dataset_0[['RT_start']] = dataset_0[['RT_start']]\
                          .fill_null(strategy='zero')



Различные способы заполнения слабо влияют на метрики по результатам части 1, но при заполнении нулями метрики для классов, которые ожидаемо выявляются по признаку `RT_start`, оказываются немного выше.

# 1. Длина ответа относительно длины стимула

In [5]:
dataset_1 = dataset_0.with_columns(
                            (pl.col('Response_annot').str.len_chars() /
                            pl.col('Stimulus').str.len_chars())\
                            .fill_null(0)\
                            .alias('relative_length')
                            )


In [6]:
from json import load

error_type_ids_file = open('../data/processed/error_type_ids.json',
                        'r', -1, 'utf-8')
error_type_ids = load(error_type_ids_file)
print(*error_type_ids.items(), sep='\n')
error_type_ids_to_names: list[str] = [''] * 8
for k, v in error_type_ids.items():
    error_type_ids_to_names[v] = k


('speech arrest', 1)
('аномия', 2)
('дизартрия', 3)
('задержка', 4)
('нет', 0)
('поиск слова', 5)
('семантическая парафазия', 6)
('фонетическая парафазия', 7)


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

X = dataset_1[['relative_length', 'RT_start']]
y = dataset_1['error_type_label']




In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
import numpy as np

def evaluate(X, y):
    # relative_length и RT_start — разного масштаба, для
    # линейных моделей здесь был бы скейлер
    pipeline_1 = Pipeline([
        ('classifier', DecisionTreeClassifier())
    ])
    k = 5
    stratified_k_fold = StratifiedKFold(n_splits = k, # test 20%
                                        shuffle = True,
                                        random_state=SEED)

    f1s = np.zeros((k, 8))
    confusion_matrices = np.zeros((k, 8, 8))

    for i, (train_mask, test_mask) in\
                    enumerate(stratified_k_fold.split(X, y)):
        X_train, X_test = X[train_mask], X[test_mask]
        y_train, y_test = y[train_mask], y[test_mask]
        pipeline_1.fit(X_train, y_train)
        predictions = pipeline_1.predict(X_test)
        f1s[i] = f1_score(y_test, predictions, average=None)
        confusion_matrices[i] = confusion_matrix(y_test, predictions)

    np.set_printoptions(suppress=True)

    print(*((name, f1.round(4).item())
            for name, f1
            in zip(error_type_ids_to_names, f1s.mean(axis=0))), sep='\n')
    
    confusion_matrix_mean = confusion_matrices.mean(axis = 0).round(2)
    print(confusion_matrix_mean)




In [38]:
evaluate(X, y)


('нет', 0.9551)
('speech arrest', 0.0)
('аномия', 0.7726)
('дизартрия', 0.1307)
('задержка', 0.51)
('поиск слова', 0.271)
('семантическая парафазия', 0.2976)
('фонетическая парафазия', 0.2465)
[[3838.4    0.2    0.8   17.8   68.8    5.    13.2   19.2]
 [   6.2    0.     0.6    0.2    0.     0.     0.     0. ]
 [   5.6    0.4   17.4    0.8    0.     0.     0.4    0.2]
 [  58.4    0.     1.4    8.2    5.6    1.     2.2    6. ]
 [ 104.2    0.     0.     5.6  109.     3.4    4.     4.4]
 [   6.2    0.     0.     1.2    3.8    4.     0.6    0. ]
 [  16.4    0.     0.     1.6    5.     0.4   10.8    2.8]
 [  39.     0.     0.     6.8    4.2    0.2    3.4   14.2]]


Модель лишь по этим двум признакам уже хорошо определяет случаи, когда ошибки нет. Также по относительной длине ответа и по задержке она неплохо определяет аномию и задержку, хотя последнюю при наличии RT_start должно быть проще определять. Как ни странно, модель не справляется со случаями speech arrest: хотя у них есть явный признак в виде нулевой длины ответа. Это может быть связано с их малым количеством в датасете и с ошибками или особенностями в разметке этих немногих примеров:

In [18]:
speech_arrest_points = dataset_1.filter(pl.col('error_type_label') == 1)
print(len(speech_arrest_points))
display(speech_arrest_points.filter(pl.col('relative_length') != 0))


35


Stimulus,RT_start,Response_annot,error_type_label,relative_length
str,i64,str,i64,f64
"""барабан""",0,"""...""",1,0.428571
"""горох""",0,"""#нрзб#""",1,1.2
"""сушить""",0,"""сушит""",1,0.833333
"""тереть""",0,"""трёт""",1,0.666667
"""черепаха""",0,"""это черепаха""",1,1.5


Видимо, некоторые типы ошибок могут быть сопряжены с характерными символами в размеченном ответе. Впрочем, на это не следует полагаться, так как решение, делающее так, не будет работать на датасете с другими стандартами разметки (например, на полученном автоматической транскрипцией).

# 2. Расстояние Левенштейна между стимулом и ответом

Сначала посмотрим на полезность доли вставок, удалений и замен. Доли, потому что стимулы могут быть разной длины, и для них будет в разной степени критично одно и то же абсолютное количество замен.

In [19]:
from rapidfuzz.distance import Levenshtein

def indelrep(stimulus, response) -> tuple[int, int, int]:
    edit_operations = Levenshtein.editops(stimulus, response)
    operation_counts = [0] * 3
    operation_ids = dict(zip(['insert', 'delete', 'replace'], range(3)))
    for tag, _, _ in edit_operations:
        operation_counts[operation_ids[tag]] += 1
    return tuple(count / len(stimulus) for count in operation_counts)

print(indelrep('роман', 'роамн'))


(0.2, 0.2, 0.0)


In [20]:
dataset_2 = dataset_1
dataset_2[['Response_annot']] =\
    dataset_2[['Response_annot']].fill_null('')


In [21]:
dataset_2 = dataset_2.with_columns(
    pl.struct(['Stimulus', 'Response_annot'])
    .map_elements(
        lambda s: indelrep(s['Stimulus'], s['Response_annot']), 
        return_dtype=pl.Struct([
            pl.Field('insertions', pl.Float64),
            pl.Field('deletions', pl.Float64),
            pl.Field('replacements', pl.Float64)
        ])
    )
    .alias('edit_distance_components')
).unnest('edit_distance_components')


In [22]:
dataset_2.sample(10)


Stimulus,RT_start,Response_annot,error_type_label,relative_length,insertions,deletions,replacements
str,i64,str,i64,f64,f64,f64,f64
"""барабан""",1078,"""это барабан""",0,1.571429,0.571429,0.0,0.0
"""копилка""",735,"""это копилка""",0,1.571429,0.571429,0.0,0.0
"""кактус""",519,"""это кактус""",0,1.666667,0.666667,0.0,0.0
"""грести""",1422,"""плывёт""",0,1.0,0.166667,0.166667,0.666667
"""скрепка""",598,"""это скрепка""",0,1.571429,0.571429,0.0,0.0
"""цепь""",700,"""это цепь""",0,2.0,1.0,0.0,0.0
"""кепка""",959,"""это кепка""",7,1.8,0.8,0.0,0.0
"""чемодан""",634,"""это чемодан""",0,1.571429,0.571429,0.0,0.0
"""стирать""",1080,"""стирает""",0,1.0,0.142857,0.142857,0.0


In [23]:
X = dataset_2[['RT_start', 'relative_length',
               'insertions', 'deletions', 'replacements']]
y = dataset_2['error_type_label']
evaluate(X, y)


('нет', 0.9547)
('speech arrest', 0.0)
('аномия', 0.769)
('дизартрия', 0.13)
('задержка', 0.508)
('поиск слова', 0.2824)
('семантическая парафазия', 0.3047)
('фонетическая парафазия', 0.249)


Как и предполагалось, стало лучше с нахождением фонетической парафазии и, в меньшей степени, дизартрии, но всё ещё не так хорошо. Без моего намерения, хотя и объяснимо, поднялась метрика для семантической парафазии и поиска слова.

